# Exercise 04 — consumer groups & manual offsets

**Goal:** understand *how Kafka tracks what you've already processed*, and what happens when a consumer crashes mid-message.

**Prerequisite:** there is data in `strom` (run [`exercise_01_produce_single.ipynb`](exercise_01_produce_single.ipynb) or [`_03_produce_batch`](exercise_03_produce_batch.ipynb)).

## Background — auto-commit vs manual commit

By default the consumer **auto-commits** its offset every 5 seconds. That's convenient — but risky:

```
1. consumer.poll() returns message X
2. you process it (insert into a database, call an API, …)
3. ❗ your process CRASHES before the next auto-commit fires
4. on restart the consumer asks: 'where were we?' → broker says X
5. message X is processed AGAIN → duplicate work
```

Or worse, the *opposite* failure: with auto-commit on, the offset can advance *before* you finish processing → on crash the message is **lost** entirely (At-Most-Once).

With `enable.auto.commit: False` **you** decide when to commit. The safe pattern is:

```
read message → process (must be idempotent or transactional) → commit
```

If processing fails the offset is never committed → the message is re-delivered → **At-Least-Once**. Combined with idempotent processing (e.g. `INSERT … ON CONFLICT DO NOTHING`), that gives you effectively exactly-once semantics.

## Step 1 — read with manual commit

Run the cell. Each iteration prints `read — processing… committed.` → that's the safe pattern in action.

In [ ]:
from confluent_kafka import Consumer

consumer = Consumer({
    'bootstrap.servers':  'redpanda:29092',
    'group.id':           'manual-commit-demo',
    'auto.offset.reset':  'earliest',
    'enable.auto.commit': False,           # we drive commits ourselves
})
consumer.subscribe(['strom'])

print(f'{"#":>3} | {"Key":>7} | {"Offset":>6} | Status')
print('-' * 45)

messages_read, empty_polls = 0, 0
while messages_read < 5 and empty_polls < 5:
    msg = consumer.poll(2.0)
    if msg is None: empty_polls += 1; continue
    if msg.error(): continue
    empty_polls = 0; messages_read += 1

    key = msg.key().decode() if msg.key() else 'None'
    print(f'{messages_read:>3} | {key:>7} | {msg.offset():>6} | '
          f'read — processing...', end='')
    # Imagine a DB INSERT here. If it raises, we never reach commit().
    consumer.commit(message=msg)   # advance the offset
    print(' committed.')

consumer.close()
print(f'\nProcessed and committed {messages_read} messages.')

## Task A — simulate a crash by skipping the commit

Modify the cell above so that `consumer.commit(message=msg)` is *not* called (comment it out). Run twice **with the same `group.id`**.

**Expected:** the second run shows you the *same* messages with the same offsets — Kafka thinks you've never processed them. Real-world consequence: you'd run your DB inserts twice. That's why all downstream sinks should be **idempotent** when using At-Least-Once.

In [ ]:
# TODO: copy the manual-commit cell above and remove the
#       consumer.commit(...) line. Run twice with the same group.id.


## Task B — two consumers, one group

Here's the magic of consumer groups: **each partition is owned by exactly one consumer in the group**. With 2 partitions and 2 consumers, the work is split. With 2 partitions and 4 consumers, two of them sit idle. With 2 partitions and 1 consumer, that one gets everything.

**To experiment:**

1. Right-click this notebook in the Explorer → *Open in New Tab* →    in *both* tabs run the cell below.
2. In a third tab, run [`exercise_03_produce_batch.ipynb`]   (exercise_03_produce_batch.ipynb).
3. **Watch:** which tab gets which messages? Why?

In [ ]:
from confluent_kafka import Consumer
from datetime import datetime

consumer = Consumer({
    'bootstrap.servers': 'redpanda:29092',
    'group.id':          'group-experiment',
    'auto.offset.reset': 'latest',         # only NEW events from now
})
consumer.subscribe(['strom', 'wasser'])
print('Listening — start the producer in another tab. Stop with the ■ button.')

try:
    while True:
        msg = consumer.poll(0.5)
        if msg is None or msg.error(): continue
        ts = datetime.now().strftime('%H:%M:%S')
        key = msg.key().decode() if msg.key() else 'None'
        print(f'[{ts}] {msg.topic()} P{msg.partition()} '
              f'offset={msg.offset()} key={key}')
except KeyboardInterrupt:
    print('\nStopped.')
finally:
    consumer.close()

## What you learned

- Auto-commit is convenient but trades correctness for simplicity.
- **At-Least-Once** = manual commit *after* processing → duplicates   on retry, no data loss.
- **At-Most-Once** = commit *before* processing → no duplicates,   possible loss.
- A consumer group **partitions the work** by partition. Adding more   consumers than partitions does *not* speed things up — the extras   sit idle.